# Colab GPU: TrackB only
Persistent dataset and SDF inputs are copied from Drive to `/content/work`; only flow training uses the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
DRIVE_RUNS = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
if not REPO.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'nibabel', 'scipy', 'scikit-image', 'matplotlib'], check=True)
assert DRIVE_RUNS.is_dir(), DRIVE_RUNS


In [ ]:
work = Path('/content/work')
dataset_src = DRIVE_RUNS / 'dataset_cache/Dataset801_IAC_LR'
cache_src = DRIVE_RUNS / 'sdf_cache_backup'
def copy_once(src, dst):
    if not dst.is_dir(): shutil.copytree(src, dst)
    print(dst, len(list(dst.iterdir())))
dataset = work / 'Dataset801_IAC_LR'
gt, coarse = work / 'gt_sdf', work / 'coarse_sdf'
copy_once(dataset_src, dataset)
copy_once(cache_src / 'gt_sdf', gt)
copy_once(cache_src / 'coarse_sdf', coarse)
splits = REPO / 'configs/splits.json'
drive_split = DRIVE_RUNS / 'configs_cache/splits.json'
assert splits.read_bytes() == drive_split.read_bytes(), 'Repo and Drive splits differ: leakage risk'
subprocess.run([sys.executable, 'data/validate_pipeline_cache.py', '--splits', str(splits), '--images', str(dataset/'imagesTr'), '--labels', str(dataset/'labelsTr'), '--gt-sdf', str(gt), '--coarse-sdf', str(coarse)], check=True)


In [ ]:
FOLD = 0
flow_out = DRIVE_RUNS / f'flow_fold{FOLD}'
subprocess.run([sys.executable, 'flow/train.py', '--config', 'configs/flow.yaml', '--splits', str(splits), '--fold', str(FOLD), '--resume', '--images', str(dataset/'imagesTr'), '--labels', str(dataset/'labelsTr'), '--gt-sdf', str(gt), '--coarse-sdf', str(coarse), '--out', str(flow_out)], check=True)
print('best checkpoint:', flow_out / 'best.pt')
